In [1]:
import numpy as np


In [2]:
test_predictions = np.load("test_predictions.npy")

In [3]:
test_predictions.shape

(740, 1)

In [4]:
test_predictions = test_predictions.squeeze()

In [25]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
import ast

In [6]:
thresholds = np.arange(0, 1, 0.1)

In [7]:
test_labels = np.load("./data/s1_data/k-folds/test/test_labels.npy")

In [8]:
# Precision: The ratio of correctly predicted positive observations to the total predicted positives
precisions = []
for threshold in thresholds:
    binary_predictions = np.where(test_predictions >= threshold, 1, 0)
    true_positives = np.sum((binary_predictions == 1) & (test_labels == 1))
    total_predicted_positive = np.sum(binary_predictions)
    precision = true_positives / total_predicted_positive if total_predicted_positive > 0 else 0
    precisions.append(precision)
    print(f"True Positives at threshold {threshold}: {true_positives}")
    print(f"Precision at threshold {threshold} = {precision}")

True Positives at threshold 0.0: 296
Precision at threshold 0.0 = 0.4
True Positives at threshold 0.1: 259
Precision at threshold 0.1 = 0.6023255813953489
True Positives at threshold 0.2: 253
Precision at threshold 0.2 = 0.6340852130325815
True Positives at threshold 0.30000000000000004: 250
Precision at threshold 0.30000000000000004 = 0.6578947368421053
True Positives at threshold 0.4: 242
Precision at threshold 0.4 = 0.6648351648351648
True Positives at threshold 0.5: 234
Precision at threshold 0.5 = 0.6685714285714286
True Positives at threshold 0.6000000000000001: 228
Precision at threshold 0.6000000000000001 = 0.6888217522658611
True Positives at threshold 0.7000000000000001: 219
Precision at threshold 0.7000000000000001 = 0.7019230769230769
True Positives at threshold 0.8: 210
Precision at threshold 0.8 = 0.7142857142857143
True Positives at threshold 0.9: 200
Precision at threshold 0.9 = 0.7434944237918215


In [10]:
all_labels = np.load("./data/s1_data/s1_flood_quality_labels.npy")
all_images = np.load("./data/s1_data/s1_resampled_quality_clipped_flood_images.npy")

In [11]:
np.sum(all_labels)

1330

In [12]:
len(all_labels)

3700

In [13]:
1330 / 3700

0.35945945945945945

In [14]:
# Recall: Ratio of correctly predicted flood labels to all true flood labels in the test set
recalls = []
total_test_positives = np.sum(test_labels)
print(f"There are {total_test_positives} positive (flooding) classes in the test set")
for threshold in thresholds:
    binary_predictions = np.where(test_predictions >= threshold, 1, 0)
    true_positives = np.sum((binary_predictions == 1) & (test_labels == 1))
    recall = true_positives / total_test_positives if total_test_positives > 0 else 0
    recalls.append(recall)
    print(f"Recall at threshold {threshold} = {recall}")

There are 296 positive (flooding) classes in the test set
Recall at threshold 0.0 = 1.0
Recall at threshold 0.1 = 0.875
Recall at threshold 0.2 = 0.8547297297297297
Recall at threshold 0.30000000000000004 = 0.8445945945945946
Recall at threshold 0.4 = 0.8175675675675675
Recall at threshold 0.5 = 0.7905405405405406
Recall at threshold 0.6000000000000001 = 0.7702702702702703
Recall at threshold 0.7000000000000001 = 0.7398648648648649
Recall at threshold 0.8 = 0.7094594594594594
Recall at threshold 0.9 = 0.6756756756756757


In [15]:
# F1 Scores: The threshold that gives the highest F1 score can be considered optimal
f1_scores = []
for i, threshold in enumerate(thresholds):
    f1_score = 2 * (precisions[i] * recalls[i]) / (precisions[i] + recalls[i])
    print(f"F1 Score for Threshold {threshold}: {f1_score}")
    f1_scores.append(f1_score)

F1 Score for Threshold 0.0: 0.5714285714285715
F1 Score for Threshold 0.1: 0.7134986225895317
F1 Score for Threshold 0.2: 0.7280575539568345
F1 Score for Threshold 0.30000000000000004: 0.7396449704142011
F1 Score for Threshold 0.4: 0.7333333333333334
F1 Score for Threshold 0.5: 0.7244582043343653
F1 Score for Threshold 0.6000000000000001: 0.7272727272727273
F1 Score for Threshold 0.7000000000000001: 0.7203947368421052
F1 Score for Threshold 0.8: 0.7118644067796611
F1 Score for Threshold 0.9: 0.7079646017699115


In [16]:
    f1_scores.append(f1_score)
# False positives at threshold 0.3
binary_predictions = np.where(test_predictions >= 0.3, 1, 0)
false_positives = np.sum((binary_predictions == 1) & (test_labels == 0))
print(f"False positives using threshold 0.3: {false_positives}")

False positives using threshold 0.3: 130


In [17]:
# Calculate accuracy from ensemble predictions
correct_predictions = np.sum(binary_predictions == test_labels)
total_predictions = len(test_labels)
accuracy = correct_predictions / total_predictions

print(f"Accuracy at 0.3: {accuracy}")

Accuracy at 0.3: 0.7621621621621621


In [18]:
# False positives at threshold 0.3
binary_predictions = np.where(test_predictions >= 0.5, 1, 0)
false_positives = np.sum((binary_predictions == 1) & (test_labels == 0))
print(f"False positives using threshold 0.5: {false_positives}")

False positives using threshold 0.5: 116


In [19]:
# Calculate accuracy from ensemble predictions
correct_predictions = np.sum(binary_predictions == test_labels)
total_predictions = len(test_labels)
accuracy = correct_predictions / total_predictions

print(f"Accuracy at 0.5: {accuracy}")

Accuracy at 0.5: 0.7594594594594595


In [20]:
# Using threshold 0.5 from here on
true_positives_indices = np.where((binary_predictions == 1) & (test_labels == 1))
true_negative_indices = np.where((binary_predictions == 0) & (test_labels == 0))
false_positives_indices = np.where((binary_predictions == 1) & (test_labels == 0))
false_negative_indices = np.where((binary_predictions == 0) & (test_labels == 1))

In [24]:
print(f"Number of false positives {false_positives_indices[0].shape}")
print(f"Number of false negatives {false_negative_indices[0].shape}")
print(f"Number of true negatives {true_negative_indices[0].shape}")
print(f"Number of true postives {true_positives_indices[0].shape}")

Number of false positives (116,)
Number of false negatives (62,)
Number of true negatives (328,)
Number of true postives (234,)


In [27]:
np.sum(test_labels)

296

In [26]:
from sklearn.metrics import precision_recall_curve, auc

precision, recall, thresholds = precision_recall_curve(test_labels, test_predictions)
auc_pr = auc(recall, precision)
print(f'Area Under the Precision-Recall Curve: {auc_pr}')

Area Under the Precision-Recall Curve: 0.7179485040587211


In [28]:
# Precision: The ratio of correctly predicted positive observations to the total predicted positives

116 + 234


350

In [29]:
234 /350

0.6685714285714286